# AI Sentinel Colab Remote Trainer

Set `BRIDGE_URL` below, then use `Runtime -> Run all`.
The notebook downloads the code bundle, current weights, and dataset from your local bridge, trains on Colab GPU, then uploads the results back to your project.

In [ ]:
BRIDGE_URL = ""
EPOCHS = 6
BATCH_SIZE = 2
INCLUDE_TEST_POSITIVES = True
UNFREEZE_BACKBONE = False

assert BRIDGE_URL.strip(), "Set BRIDGE_URL before running the notebook"

In [ ]:
import json
import os
import shutil
import subprocess
import urllib.request
from pathlib import Path

ROOT = Path('/content/ai-sentinel-colab')
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True, exist_ok=True)

manifest = json.loads(urllib.request.urlopen(f"{BRIDGE_URL}/manifest").read().decode('utf-8'))
token = manifest['token']

def download(name, target):
    url = f"{BRIDGE_URL}/download/{name}"
    print('Downloading', url)
    urllib.request.urlretrieve(url, target)
    return target

bundle_zip = download(manifest['bundle']['filename'], ROOT / manifest['bundle']['filename'])
dataset_zip = download(manifest['dataset']['filename'], ROOT / manifest['dataset']['filename'])
weights_path = download(manifest['weights']['filename'], ROOT / manifest['weights']['filename'])

import zipfile
with zipfile.ZipFile(bundle_zip) as zf:
    zf.extractall(ROOT)

print('Workspace ready at', ROOT)

In [ ]:
subprocess.run(['pip', 'install', '-r', str(ROOT / 'backend' / 'requirements-train.txt')], check=True)
print('Dependencies installed')

In [ ]:
train_cmd = [
    'python', '-m', 'backend.train_finetune',
    '--sources', str(dataset_zip),
    '--weights', str(weights_path),
    '--output', str(ROOT / 'backend' / 'best_model.pt'),
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--cache-dir', str(ROOT / 'cache'),
]
if INCLUDE_TEST_POSITIVES:
    train_cmd.append('--include-test-positives')
if UNFREEZE_BACKBONE:
    train_cmd.append('--unfreeze-backbone')

train_proc = subprocess.run(train_cmd, cwd=str(ROOT), text=True, capture_output=True)
print(train_proc.stdout)
if train_proc.returncode != 0:
    print(train_proc.stderr)
    raise RuntimeError(f"train_finetune failed with code {train_proc.returncode}")

calib_cmd = [
    'python', '-m', 'backend.train_calibration',
    '--dataset-zip', str(dataset_zip),
    '--weights', str(ROOT / 'backend' / 'best_model.pt'),
    '--output', str(ROOT / 'backend' / 'model_calibration.json'),
]
calib_proc = subprocess.run(calib_cmd, cwd=str(ROOT), text=True, capture_output=True)
print(calib_proc.stdout)
if calib_proc.returncode != 0:
    print(calib_proc.stderr)
    raise RuntimeError(f"train_calibration failed with code {calib_proc.returncode}")

print('Training finished')

In [ ]:
import urllib.request

report = {
    'status': 'ok',
    'epochs': EPOCHS,
    'batchSize': BATCH_SIZE,
    'includeTestPositives': INCLUDE_TEST_POSITIVES,
    'unfreezeBackbone': UNFREEZE_BACKBONE,
}
(ROOT / 'backend' / 'training_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')

def upload(path):
    target = path.name
    req = urllib.request.Request(
        f"{BRIDGE_URL}/upload/{target}?token={token}",
        data=path.read_bytes(),
        method='POST',
        headers={'Content-Type': 'application/octet-stream'},
    )
    with urllib.request.urlopen(req) as resp:
        print(target, resp.read().decode('utf-8'))

upload(ROOT / 'backend' / 'best_model.pt')
upload(ROOT / 'backend' / 'model_calibration.json')
upload(ROOT / 'backend' / 'training_report.json')
print('Artifacts uploaded back to the local project')